In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

from replay_buffer import calculate_lambda_returns, LambdaReplayBuffer

print("calculate_lambda_returns (njit), LambdaReplayBuffer — imported from replay_buffer.")

In [6]:

import numpy as np
import torch as th

from rainbow_dqn import RainbowDQN, DuelingDQNPolicy


class LambdaRainbowDQN(RainbowDQN):
    """
    RainbowDQN + LambdaReplayBuffer wiring. One override only: _on_step.

    _on_step — cache refresh
        Mirrors DQN._on_step's polyak condition exactly:
          self._n_calls % max(target_update_interval // n_envs, 1) == 0
        At that frequency: refresh_cache(q_net) — rebuilds λ-returns using
        the online Q-network immediately after its polyak update.
        No-op when replay_buffer is not a LambdaReplayBuffer.

    train() is inherited from RainbowDQN unchanged.
    LambdaReplayBuffer.sample() returns discounts=0, so:
      target_q = rewards + (1-dones) * 0 * next_q = lambda_returns.

    Watkins mask is recomputed at refresh time by LambdaReplayBuffer itself —
    no collection-time greedy flag is needed here.

    Requirement: learning_starts >= target_update_interval
        Ensures at least one refresh_cache fires before train() is first called.

    Composition with StickyDQN:
      class LambdaStickyDQN(LambdaRainbowDQN, StickyDQN): pass
      MRO: LambdaRainbowDQN → StickyDQN → RainbowDQN → DQN
      StickyDQN.predict() and _bellman_next_q() resolve correctly via MRO.
    """

    def _on_step(self) -> None:
        super()._on_step()   # polyak update + ε schedule (increments self._n_calls)

        if not isinstance(self.replay_buffer, LambdaReplayBuffer):
            return

        if self._n_calls % max(self.target_update_interval // self.n_envs, 1) == 0:
            self.replay_buffer.refresh_cache(self.q_net, self.device)


print("LambdaRainbowDQN — defined.")


LambdaRainbowDQN — defined.


In [ ]:

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))
# reference implementation is archived; patch path for backward compat
sys.path.insert(0, str(Path.cwd().parent / "archive" / "other"))
from return_calculation import calculate_lambda_returns as ref_lambda

import gymnasium as gym
import numpy as np
import torch as th
from stable_baselines3 import DQN
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.evaluation import evaluate_policy

from replay_buffer import (
    LambdaReplayBuffer,
    PrioritizedReplayBuffer,
    calculate_lambda_returns,
)

In [8]:
ours[i]

array([-0.4190439 , -2.16528   , -2.1123655 , -0.8621399 , -1.4556544 ,
        1.2517762 ,  0.21372087, -0.8113653 , -0.81819355,  0.9285029 ,
       -2.0120366 , -1.4504625 , -1.4222646 , -0.6348132 , -0.69688386,
        0.9242179 , -0.7317546 , -1.371998  ,  0.02512871,  0.16593617,
        0.36504495, -0.01293877, -1.4228626 , -0.946941  ,  0.46130905,
        0.74180406, -1.4965374 ,  0.8565284 ,  0.8367561 , -0.24396737,
        0.22523575, -0.5079928 ,  0.10997764, -3.2176344 , -2.2573545 ,
       -0.9590373 , -2.4992692 , -1.9879218 , -1.9119558 ,  0.7820414 ,
       -0.84114915, -0.7295772 ,  0.80764073,  1.5819377 ,  1.3630456 ,
        1.6671019 ,  1.5229933 , -0.04561153,  1.5681144 ,  1.7521087 ],
      dtype=float32)

In [9]:

# ── LambdaReplayBuffer vs PrioritizedReplayBuffer ────────────────────────────
#
# Both inherit NStepReplayBuffer.
# Storage layout (SB3 VecEnv convention, shared):
#   observations  (buffer_size, n_envs, *obs_shape)
#   actions       (buffer_size, n_envs, 1)
#   rewards       (buffer_size, n_envs)
#   dones         (buffer_size, n_envs)
#
# ┌──────────────────────┬──────────────────────────┬──────────────────────────┐
# │                      │ PrioritizedReplayBuffer  │ LambdaReplayBuffer       │
# ├──────────────────────┼──────────────────────────┼──────────────────────────┤
# │ Sampling unit        │ (row, env) pairs         │ contiguous block,        │
# │                      │  independent             │  one env per block       │
# ├──────────────────────┼──────────────────────────┼──────────────────────────┤
# │ Priority             │ |TD error|^α  (SumTree)  │ uniform (over cache)     │
# ├──────────────────────┼──────────────────────────┼──────────────────────────┤
# │ Bellman target       │ computed in train()      │ precomputed in cache     │
# │                      │  r + γ·next_q            │  G^λ  packed as rewards  │
# ├──────────────────────┼──────────────────────────┼──────────────────────────┤
# │ discounts in sample()│ γ  (scalar / tensor)     │ 0  (tensor of zeros)     │
# │ target_q in train()  │ r + (1-d)·γ·next_q      │ r + (1-d)·0  =  G^λ     │
# ├──────────────────────┼──────────────────────────┼──────────────────────────┤
# │ Extra overhead       │ SumTree O(log B)/step    │ cache rebuild at polyak  │
# │                      │ priority update per batch│  freq — one fwd pass     │
# └──────────────────────┴──────────────────────────┴──────────────────────────┘
#
# ── n_envs = 8 ───────────────────────────────────────────────────────────────
#
# PrioritizedReplayBuffer with n_envs=8:
#   SumTree capacity = buffer_size  (row indices, not transitions)
#   sample() draws row indices → _get_samples picks a random env per row
#   One priority covers all 8 envs' transitions in that row  (coarse granularity)
#   Buffer fills 8× faster but SumTree size is unchanged
#
# LambdaReplayBuffer with n_envs=8:
#   Buffer fills 8× faster  (8 transitions written per pos increment)
#   Each block is tied to exactly one env:
#     env_indices ~ Uniform(0, 8)  in _sample_block_starts
#   is_greedy shape: (buffer_size, 8) — per (pos, env) flag
#
#   Expected blocks per env per refresh:
#     n_blocks = cache_size // block_size    e.g. 10_000 // 200 = 50
#     ~50 / 8 ≈ 6 blocks per env  (balanced on average)
#
#   Recommended cache_size scaling:
#     cache_size ≥ block_size × n_envs × desired_blocks_per_env
#     e.g. 200 × 8 × 8 = 12_800  (must be divisible by block_size)
#
#   Refresh cadence  (target_update_interval=500, n_envs=8):
#     every  max(500 // 8, 1) = 62  _n_calls  →  62 × 8 = 496  transitions
#     ≈ target_update_interval  (exact mirror of polyak update timing)
#
# Combining PER + Lambda:
#   PER samples rows; Lambda samples blocks — the two mechanisms conflict.
#   Correct combination: after refresh_cache, compute |G^λ − Q(s,a)| per
#   cached transition and write priorities to the SumTree.
#   Daley & Amato use a simpler median-based weight inside the cache instead.
#   Multiple inheritance  LambdaReplayBuffer(PrioritizedReplayBuffer)  would
#   require overriding sample() to also update the SumTree — non-trivial.

print("Structural comparison — see comments above.")


Structural comparison — see comments above.


In [10]:

import time
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3.common.vec_env import DummyVecEnv

from replay_buffer import PrioritizedReplayBuffer

# ── Config ────────────────────────────────────────────────────────────────────
N_ENVS     = 8
OBS_DIM    = 10
N_ACTIONS  = 3
BUF_SIZE   = 2_000
BLOCK_SIZE = 50
CACHE_SIZE = 800      # 16 blocks; ≥ block_size × n_envs × 2
N_STEPS    = 1
GAMMA      = 0.99
BATCH_SIZE = 256
DEVICE     = th.device("cpu")


class ToyEnv(gym.Env):
    """Minimal 3-action env matching trading action space."""
    observation_space = spaces.Box(-1., 1., (OBS_DIM,), dtype=np.float32)
    action_space      = spaces.Discrete(N_ACTIONS)
    def reset(self, **kw): return self.observation_space.sample(), {}
    def step(self, a):     return self.observation_space.sample(), 0., False, False, {}


vec_env = DummyVecEnv([ToyEnv] * N_ENVS)

# ── Instantiate ───────────────────────────────────────────────────────────────
lambda_buf = LambdaReplayBuffer(
    buffer_size       = BUF_SIZE,
    observation_space = vec_env.observation_space,
    action_space      = vec_env.action_space,
    device            = DEVICE,
    n_envs            = N_ENVS,
    n_steps           = N_STEPS,
    gamma             = GAMMA,
    lambd             = 0.8,
    block_size        = BLOCK_SIZE,
    cache_size        = CACHE_SIZE,
    use_watkins       = True,
    per_priority      = 0.5,
    per_total_samples = 1_000_000,
)

per_buf = PrioritizedReplayBuffer(
    buffer_size       = BUF_SIZE,
    observation_space = vec_env.observation_space,
    action_space      = vec_env.action_space,
    device            = DEVICE,
    n_envs            = N_ENVS,
    n_steps           = N_STEPS,
    gamma             = GAMMA,
    alpha             = 0.6,
    beta              = 0.4,
    total_samples     = 1_000_000,
)

# ── Fill with random transitions ──────────────────────────────────────────────
OBS    = np.zeros((N_ENVS, OBS_DIM), dtype=np.float32)
N_FILL = BLOCK_SIZE + 10    # minimum for _sample_block_starts

for _ in range(N_FILL):
    next_obs = np.random.randn(N_ENVS, OBS_DIM).astype(np.float32)
    actions  = np.random.randint(0, N_ACTIONS, size=(N_ENVS,))
    rewards  = np.random.randn(N_ENVS).astype(np.float32)
    dones    = np.zeros(N_ENVS, dtype=bool)
    infos    = [{} for _ in range(N_ENVS)]
    lambda_buf.add(OBS, next_obs, actions, rewards, dones, infos)
    per_buf.add(OBS, next_obs, actions, rewards, dones, infos)
    OBS = next_obs

print(f"pos (rows written):    {lambda_buf.pos}")
print(f"transitions stored:    {lambda_buf.pos * N_ENVS}  (pos × n_envs)")

# ── Lambda: refresh_cache + sample ───────────────────────────────────────────
q_net = th.nn.Sequential(
    th.nn.Linear(OBS_DIM, 64), th.nn.ReLU(),
    th.nn.Linear(64, N_ACTIONS),
)

t0 = time.perf_counter()
lambda_buf.refresh_cache(q_net, DEVICE)
t_refresh = (time.perf_counter() - t0) * 1e3

print(f"\nLambdaReplayBuffer — refresh_cache  ({t_refresh:.1f} ms)")
print(f"  _cache_obs:     {lambda_buf._cache_obs.shape}")
print(f"  _cache_actions: {lambda_buf._cache_actions.shape}")
print(f"  _cache_returns: {lambda_buf._cache_returns.shape}")

# Priority distribution
p_now      = lambda_buf.priority_p
above_med  = (lambda_buf._slot_probs > 1.0 / CACHE_SIZE).sum()
below_med  = (lambda_buf._slot_probs < 1.0 / CACHE_SIZE).sum()
at_med     = CACHE_SIZE - above_med - below_med
print(f"\n  Priority (Daley & Amato §3.3):")
print(f"    p_now       = {p_now:.3f}  (anneals {lambda_buf.per_priority} → 0 over {lambda_buf.per_total_samples:,} steps)")
print(f"    slots above median: {above_med:4d}  prob = {(1+p_now)/CACHE_SIZE:.6f}")
print(f"    slots at median:    {at_med:4d}  prob = {1/CACHE_SIZE:.6f}")
print(f"    slots below median: {below_med:4d}  prob = {(1-p_now)/CACHE_SIZE:.6f}")
print(f"    ratio high/low = {(1+p_now)/(1-p_now):.2f}x  (1.0x when fully annealed)")

b_lam = lambda_buf.sample(BATCH_SIZE)
print(f"\n  sample({BATCH_SIZE}):")
print(f"    observations: {tuple(b_lam.observations.shape)}")
print(f"    rewards:      {tuple(b_lam.rewards.shape)}   ← G^λ (precomputed target)")
print(f"    discounts:    {b_lam.discounts.sum().item():.0f}   ← kills bootstrap in train()")
print(f"    dones:        {b_lam.dones.sum().item():.0f}")

# ── PER: sample ───────────────────────────────────────────────────────────────
b_per = per_buf.sample(BATCH_SIZE)
print(f"\nPrioritizedReplayBuffer — sample({BATCH_SIZE}):")
print(f"  observations: {tuple(b_per.observations.shape)}")
print(f"  rewards:      {tuple(b_per.rewards.shape)}   ← n-step accumulated reward")
print(f"  discounts:    {b_per.discounts.unique().tolist()}   ← γ^n_steps, passed to train()")
print(f"  weights:      min={b_per.weights.min():.3f}  max={b_per.weights.max():.3f}  (IS correction)")
print(f"  indices:      {b_per.indices.shape}   ← row indices → _update_priorities")

# ── n_envs=8: block distribution across envs ─────────────────────────────────
n_blocks = CACHE_SIZE // BLOCK_SIZE
_, sampled_env_indices = lambda_buf._sample_block_starts(n_blocks)
counts = np.bincount(sampled_env_indices, minlength=N_ENVS)
print(f"\nn_envs={N_ENVS}  →  {n_blocks} blocks  →  expected ~{n_blocks // N_ENVS} per env")
print(f"  actual blocks per env: {counts}")
print(f"  recommended cache_size for ~8 blocks/env: "
      f"{BLOCK_SIZE * N_ENVS * 8}  (= block_size × n_envs × 8)")


pos (rows written):    60
transitions stored:    480  (pos × n_envs)

LambdaReplayBuffer — refresh_cache  (0.7 ms)
  _cache_obs:     (800, 10)
  _cache_actions: (800,)
  _cache_returns: (800,)

  Priority (Daley & Amato §3.3):
    p_now       = 0.500  (anneals 0.5 → 0 over 1,000,000 steps)
    slots above median:  399  prob = 0.001875
    slots at median:       0  prob = 0.001250
    slots below median:  401  prob = 0.000625
    ratio high/low = 3.00x  (1.0x when fully annealed)

  sample(256):
    observations: (256, 10)
    rewards:      (256, 1)   ← G^λ (precomputed target)
    discounts:    0   ← kills bootstrap in train()
    dones:        0

PrioritizedReplayBuffer — sample(256):
  observations: (256, 10)
  rewards:      (256, 1)   ← n-step accumulated reward
  discounts:    [0.9900000095367432]   ← γ^n_steps, passed to train()
  weights:      min=1.000  max=1.000  (IS correction)
  indices:      (256,)   ← row indices → _update_priorities

n_envs=8  →  16 blocks  →  expected ~2 